# 第18章 风险管理系统 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch18_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch18_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：三法 VaR/CVaR + 图18-1


In [ ]:
import numpy as np
from fi import var as v, plotting
plotting.use_chinese_style()
sigma = v.bond_pnl_sigma(1e8, 5, 0.0005)
rng = np.random.default_rng(7)
normal = rng.normal(0,sigma,100000); fat = rng.standard_t(4,100000)*sigma/np.sqrt(4/2)
print(f'参数法(正态) 99%VaR={v.parametric_var(sigma,0.99):,.0f}')
mc_var, mc_cvar = v.monte_carlo_var(sigma,0.99,n=200000,seed=1)
print(f'蒙特卡洛 99%VaR={mc_var:,.0f} CVaR={mc_cvar:,.0f}')
print(f'历史法(厚尾) 99%VaR={v.historical_var(fat,0.99):,.0f} CVaR={v.historical_cvar(fat,0.99):,.0f}  <- 厚尾更大')
var99, cvar99 = v.parametric_var(sigma,0.99), v.parametric_cvar(sigma,0.99)
fig, ax = plotting.new_axes()
ax.hist(normal/1e4, bins=120, density=True, alpha=0.5, label='正态'); ax.hist(fat/1e4, bins=200, density=True, alpha=0.4, label='厚尾')
ax.axvline(-var99/1e4, color='C3', ls='--', label=f'VaR≈{var99/1e4:.0f}万'); ax.axvline(-cvar99/1e4, color='C1', ls=':', label=f'CVaR≈{cvar99/1e4:.0f}万')
ax.set_xlim(-150,150); ax.set_xlabel('日损益(万元)'); ax.set_ylabel('密度'); ax.set_title('损益分布与VaR/CVaR'); ax.legend(); fig.tight_layout()


## 编程实验 8：压力情景表


In [ ]:
scenarios = {'+100bp 平移':(0.01,5,30), '+200bp 平移':(0.02,5,30), '曲线变陡(长端+50bp)':(0.005,7,30), '信用利差+150bp':(0.015,5,20)}
print('情景                组合损益(万元)')
for name,(dy,D,C) in scenarios.items():
    pnl = v.scenario_pnl(1e8, D, C, dy)/1e4
    print(f'{name:<20} {pnl:>10.1f}')


## 编程实验 9：综合全书——组合风险仪表盘


In [ ]:
# 组合：国债 + 信用债 + 可赎回债 + 互换，汇总风险指标
from fi.cashflow import make_cashflows
from fi import risk, credit, tree, swap
from fi.pricing import price_bond
rows = []
cf,t = make_cashflows(0.0255,10,2,100); rows.append(('10Y国债', price_bond(cf,t,0.0255,2), risk.modified_duration(cf,t,0.0255,2)))
cf2,t2 = make_cashflows(0.045,5,2,100); rows.append(('5Y信用债', price_bond(cf2,t2,0.045,2), risk.modified_duration(cf2,t2,0.045,2)))
tr = tree.short_rate_tree(0.03,0.20,6); rows.append(('可赎回债', tree.value_bond(tr,6.0,100,call_price=100,call_from=1), tree.effective_duration_tree(0.03,0.20,6,6.0,call_price=100,call_from=1)))
dfs=[(1.03)**-x for x in range(1,6)]; rows.append(('5Y payer互换', swap.swap_value(0.025,dfs,[1]*5,100,True), swap.swap_dv01(dfs,[1]*5,100)/1e-4/100))
print(f"{'持仓':<14}{'价值/价格':>12}{'久期':>10}")
for nm,val,dur in rows: print(f'{nm:<14}{val:>12.4f}{dur:>10.4f}')
# 组合层 VaR（示意：等市值、用平均久期）
avg_dur = np.mean([r[2] for r in rows[:3]])
sig = v.bond_pnl_sigma(3e8, avg_dur, 0.0005)
print(f'\n组合(3亿,均久期{avg_dur:.2f}) 99% 1日 VaR ≈ {v.parametric_var(sig,0.99):,.0f} 元')
print('这就是全书工具的风险集成：定价(pricing)+久期(risk)+信用(credit)+含权(tree)+互换(swap)+VaR(var)')
